In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxinPred)

This notebook processes and standardizes the **ToxinPred** dataset into a clean, model-ready format. The raw data are provided as multiple plain-text/CSV files where each file contains peptide sequences and \
 the **class label is inferred from the filename** (e.g., `*neg*` → non-toxic, otherwise toxic).

- **Toxic effect / endpoint:** toxic
- **Source:** ToxiPred
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw sequence files** from the ToxinPred folder and concatenates them into a single table.
- **Infers binary labels from filenames**:
  - sequences from files whose name contains `neg` are labeled as `0` (non-toxic),
  - all other files are labeled as `1` (toxic).
- **Standardizes the schema** to two columns: `sequence` and `label`.
- **Performs duplicate sequence quality control**:
  - keeps one entry per unique sequence when labels are consistent,
  - flags sequences that appear with conflicting labels as erroneous.
- **Builds dataset metadata** using the centralized `raw_data_description.xlsx`.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv` (deduplicated, consistent labels),
  - `detected_error_sequences.csv` (conflicting duplicates),
  - `metadata.json` (dataset provenance and summary statistics).

In [2]:
name_source = "ToxinPred"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*"):
    df = pd.read_csv(file, header=None, names=["sequence"])
    df["source_file"] = file.name
    dfs.append(df)
df_toxinpred = pd.concat(dfs, ignore_index=True)

In [4]:
df_toxinpred = (
    df_toxinpred
    .assign(
        label=lambda d: d["source_file"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df_toxinpred.shape

(21650, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxinpred, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(19391, 2)

In [7]:
df_errors.shape

(24, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxinpred)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2013,
 'last update date': datetime.datetime(2013, 9, 13, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'txt',
 'peptide property': 'toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot;No information',
 'repository or server': 'http://crdd.osdd.net/raghava/toxinpred/',
 'publication': 'https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0073957#s2',
 'number_of_raw_sequences': 21650,
 'number_of_sequences_retained': 19391,
 'number_of_positive_sequences': 2084,
 'number_of_negative_sequences': 17307,
 'number_of_erroneous_sequences': 24,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)